In [ ]:
!nvidia-smi
!python --version
!df -h

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y \
    git \
    wget \
    curl \
    build-essential \
    cmake \
    pkg-config \
    ffmpeg \
    libavformat-dev \
    libavcodec-dev \
    libavdevice-dev \
    libavutil-dev \
    libswscale-dev \
    libswresample-dev \
    libavfilter-dev \
    libgl1-mesa-dev \
    libegl1 \
    libosmesa6-dev \
    patchelf

In [ ]:
!which cmake
!cmake --version
!ffmpeg -version | head -3

In [ ]:
%cd /kaggle/working

!wget -q "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh" -O Miniforge3-Linux-x86_64.sh
!bash Miniforge3-Linux-x86_64.sh -b -p /kaggle/working/miniforge3

In [ ]:
!source /kaggle/working/miniforge3/etc/profile.d/conda.sh && \
conda create -y -n lerobot python=3.12

In [ ]:
!source /kaggle/working/miniforge3/etc/profile.d/conda.sh && \
conda activate lerobot && \
python --version && \
which python

In [ ]:
%cd /kaggle/working

!git clone https://github.com/huggingface/lerobot.git
%cd /kaggle/working/lerobot

In [ ]:
!mkdir -p /kaggle/working/build_bin
!ln -sf /usr/bin/cmake /kaggle/working/build_bin/cmake
!/kaggle/working/build_bin/cmake --version

In [ ]:
%cd /kaggle/working/lerobot

!source /kaggle/working/miniforge3/etc/profile.d/conda.sh && \
conda activate lerobot && \
unset PYTHONPATH && \
export PATH="/kaggle/working/build_bin:$CONDA_PREFIX/bin:/usr/bin:/bin:$PATH" && \
python -m pip install --upgrade pip setuptools wheel && \
python -m pip install -e . && \
python -m pip install -e ".[smolvla]" && \
python -m pip install -e ".[libero]" --no-cache-dir

In [ ]:
%cd /kaggle/working/lerobot

! /kaggle/working/miniforge3/envs/lerobot/bin/python -m pip install mujoco robosuite
! /kaggle/working/miniforge3/envs/lerobot/bin/python -m pip install -e ".[libero]" --no-cache-dir

In [ ]:
!ls -lh /kaggle/working/miniforge3/envs/lerobot/bin/python
!/kaggle/working/miniforge3/envs/lerobot/bin/python --version
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import sys; print(sys.executable)"

In [ ]:
%cd /kaggle/working/lerobot

!/kaggle/working/miniforge3/envs/lerobot/bin/python -m pip show lerobot
!/kaggle/working/miniforge3/envs/lerobot/bin/python -m pip show mujoco
!/kaggle/working/miniforge3/envs/lerobot/bin/python -m pip show robosuite
!/kaggle/working/miniforge3/envs/lerobot/bin/python -m pip show hf-libero

In [ ]:
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import sys; print('Python executable:', sys.executable)"
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import torch; print('Torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA')"
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import lerobot; print('LeRobot OK')"
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import mujoco; print('MuJoCo OK')"
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import robosuite; print('robosuite OK')"
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import libero; print('LIBERO OK')"

In [ ]:
%cd /kaggle/working

!mkdir -p vla_robotic_manipulation/scripts
!mkdir -p vla_robotic_manipulation/logs
!mkdir -p vla_final_demo/media/final_demo
!mkdir -p vla_final_demo/outputs/eval
!mkdir -p vla_final_demo/results

In [ ]:
%cd /kaggle/working/lerobot

!/kaggle/working/miniforge3/envs/lerobot/bin/python -m pip install -e ".[pi]"

In [ ]:
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import lerobot; print('LeRobot OK')"

In [ ]:
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "from huggingface_hub import repo_exists; repo_id='lerobot/pi05_libero'; print(repo_id, 'exists:', repo_exists(repo_id))"

In [ ]:
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "from huggingface_hub import repo_exists; repo_id='lerobot/pi05_libero_finetuned'; print(repo_id, 'exists:', repo_exists(repo_id))"

In [ ]:
%cd /kaggle/working/lerobot

!/kaggle/working/miniforge3/envs/lerobot/bin/python -m pip install -e ".[pi]"

In [ ]:
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "import lerobot; print('LeRobot OK')"

In [ ]:
%%writefile /kaggle/working/hf_login.py
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token, add_to_git_credential=False)

info = whoami()
print("Logged in to Hugging Face as:", info["name"])

In [ ]:
%%writefile /kaggle/working/hf_login_bridge.py
import os
import subprocess
import sys
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")

if token is None or len(token.strip()) == 0:
    raise RuntimeError("HF_TOKEN secret is empty or not enabled for this notebook.")

env = os.environ.copy()
env["HF_TOKEN"] = token

cmd = [
    "/kaggle/working/miniforge3/envs/lerobot/bin/python",
    "-c",
    """
import os
from huggingface_hub import login, whoami

token = os.environ["HF_TOKEN"]
login(token=token, add_to_git_credential=False)
info = whoami()
print("Logged in to Hugging Face as:", info["name"])
"""
]

result = subprocess.run(cmd, env=env, text=True)
sys.exit(result.returncode)

In [ ]:
!python /kaggle/working/hf_login_bridge.py

In [ ]:
!/kaggle/working/miniforge3/envs/lerobot/bin/python -c "from transformers import AutoTokenizer; AutoTokenizer.from_pretrained('google/paligemma-3b-pt-224'); print('PaliGemma access OK')"

In [ ]:
!mkdir -p /kaggle/working/vla_robotic_manipulation/scripts
!mkdir -p /kaggle/working/vla_robotic_manipulation/results
!mkdir -p /kaggle/working/vla_robotic_manipulation/logs
!mkdir -p /kaggle/working/vla_robotic_manipulation/media/final_demo
!mkdir -p /kaggle/working/vla_robotic_manipulation/outputs/eval

!ls -lh /kaggle/working/vla_robotic_manipulation

In [ ]:
%%writefile /kaggle/working/vla_robotic_manipulation/scripts/run_vla.py
import argparse
import csv
import os
import re
import shutil
import subprocess
import sys
from difflib import SequenceMatcher
from pathlib import Path

from libero.libero.benchmark import get_benchmark


SUPPORTED_SUITES = [
    "libero_spatial",
    "libero_object",
    "libero_goal",
    "libero_10",
]


def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


def safe_get(obj, names, default=""):
    for name in names:
        if hasattr(obj, name):
            value = getattr(obj, name)
            if value is not None:
                return value
    return default


def load_tasks_for_suite(suite_name: str):
    benchmark_cls = get_benchmark(suite_name)
    benchmark = benchmark_cls()

    n_tasks = getattr(benchmark, "n_tasks", None)
    if n_tasks is None:
        n_tasks = len(getattr(benchmark, "tasks", []))

    rows = []

    for task_id in range(n_tasks):
        task = benchmark.get_task(task_id)

        task_name = safe_get(
            task,
            ["name", "task_name", "problem_name"],
            default=f"task_{task_id}",
        )

        instruction = safe_get(
            task,
            ["language", "language_instruction", "instruction", "description"],
            default="",
        )

        bddl_file = safe_get(
            task,
            ["bddl_file", "bddl_file_name"],
            default="",
        )

        rows.append(
            {
                "suite": suite_name,
                "task_id": task_id,
                "task_name": str(task_name),
                "instruction": str(instruction),
                "bddl_file": str(bddl_file),
            }
        )

    return rows


def load_all_tasks():
    all_rows = []
    for suite in SUPPORTED_SUITES:
        try:
            all_rows.extend(load_tasks_for_suite(suite))
        except Exception as exc:
            print(f"[WARNING] Could not load suite {suite}: {exc}")
    return all_rows


def print_tasks(rows):
    print("=" * 100)
    print("Available supported natural-language task commands")
    print("=" * 100)

    current_suite = None
    for row in rows:
        if row["suite"] != current_suite:
            current_suite = row["suite"]
            print()
            print(f"SUITE: {current_suite}")
            print("-" * 100)

        print(f"[{row['task_id']}] {row['instruction']}")
        print(f"    task_name: {row['task_name']}")
        print(f"    bddl_file: {row['bddl_file']}")

    print("=" * 100)


def save_task_list(rows, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / "supported_libero_tasks.csv"
    txt_path = output_dir / "supported_libero_tasks.txt"

    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["suite", "task_id", "task_name", "instruction", "bddl_file"],
        )
        writer.writeheader()
        writer.writerows(rows)

    with txt_path.open("w") as f:
        for row in rows:
            f.write(f"Suite       : {row['suite']}\n")
            f.write(f"Task ID     : {row['task_id']}\n")
            f.write(f"Instruction : {row['instruction']}\n")
            f.write(f"Task name   : {row['task_name']}\n")
            f.write(f"BDDL file   : {row['bddl_file']}\n")
            f.write("-" * 100 + "\n")

    print(f"Saved task CSV: {csv_path}")
    print(f"Saved task TXT: {txt_path}")


def match_instruction(rows, instruction: str, min_score: float = 0.65):
    query = normalize_text(instruction)

    exact_matches = [
        row for row in rows if normalize_text(row["instruction"]) == query
    ]
    if exact_matches:
        return exact_matches[0], 1.0

    scored = []
    for row in rows:
        target = normalize_text(row["instruction"])
        score = SequenceMatcher(None, query, target).ratio()
        scored.append((score, row))

    scored.sort(reverse=True, key=lambda x: x[0])
    best_score, best_row = scored[0]

    if best_score < min_score:
        print()
        print("No confident task match found.")
        print(f"Best score: {best_score:.3f}")
        print()
        print("Closest supported commands:")
        for score, row in scored[:5]:
            print(f"score={score:.3f} | suite={row['suite']} | task_id={row['task_id']} | {row['instruction']}")
        raise SystemExit(2)

    return best_row, best_score


def find_lerobot_eval():
    kaggle_path = Path("/kaggle/working/miniforge3/envs/lerobot/bin/lerobot-eval")
    if kaggle_path.exists():
        return str(kaggle_path)
    return "lerobot-eval"


def run_eval(row, args):
    project_dir = Path(args.project_dir).resolve()
    output_dir = project_dir / "outputs" / "eval" / f"{args.policy_tag}_{row['suite']}_task_{row['task_id']}_{args.device}"
    log_dir = project_dir / "logs"
    media_dir = project_dir / "media" / "final_demo"

    output_dir.mkdir(parents=True, exist_ok=True)
    log_dir.mkdir(parents=True, exist_ok=True)
    media_dir.mkdir(parents=True, exist_ok=True)

    log_file = log_dir / f"{args.policy_tag}_{row['suite']}_task_{row['task_id']}_{args.device}.log"
    final_video = media_dir / f"{args.policy_tag}_{row['suite']}_task_{row['task_id']}_{args.device}.mp4"

    lerobot_eval = find_lerobot_eval()

    env = os.environ.copy()
    env["MUJOCO_GL"] = "egl"
    env["PYOPENGL_PLATFORM"] = "egl"
    env["MPLBACKEND"] = "Agg"

    cmd = [
        lerobot_eval,
        f"--output_dir={output_dir}",
        f"--policy.path={args.policy}",
        "--policy.n_action_steps=10",
        "--env.type=libero",
        f"--env.task={row['suite']}",
        f"--env.task_ids=[{row['task_id']}]",
        "--env.max_parallel_tasks=1",
        "--eval.batch_size=1",
        f"--eval.n_episodes={args.episodes}",
        "--eval.use_async_envs=false",
        f"--policy.device={args.device}",
    ]

    if args.dtype:
        cmd.append(f"--policy.dtype={args.dtype}")

    print("=" * 100)
    print("Terminal-driven VLA robotic manipulation demo")
    print("=" * 100)
    print(f"Typed instruction : {args.instruction}")
    print(f"Matched suite     : {row['suite']}")
    print(f"Matched task ID   : {row['task_id']}")
    print(f"Supported command : {row['instruction']}")
    print(f"Task name         : {row['task_name']}")
    print(f"Policy            : {args.policy}")
    print(f"Device            : {args.device}")
    print(f"Output directory  : {output_dir}")
    print(f"Log file          : {log_file}")
    print(f"Final video       : {final_video}")
    print("=" * 100)
    print("Running evaluation...")
    print("Command:")
    print(" ".join(cmd))
    print("=" * 100)

    with log_file.open("w") as log:
        process = subprocess.Popen(
            cmd,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=env,
            cwd=project_dir,
        )

        # Answer LIBERO's dataset-path prompt if it appears.
        try:
            process.stdin.write("N\n")
            process.stdin.flush()
        except Exception:
            pass

        for line in process.stdout:
            print(line, end="")
            log.write(line)

        return_code = process.wait()

    if return_code != 0:
        print(f"Evaluation failed with return code {return_code}")
        print(f"See log: {log_file}")
        raise SystemExit(return_code)

    videos = sorted(output_dir.rglob("*.mp4"))
    if not videos:
        print("No video file was generated.")
        print(f"Check output directory: {output_dir}")
        raise SystemExit(3)

    shutil.copy2(videos[0], final_video)

    print("=" * 100)
    print("Demo completed successfully.")
    print(f"Generated rollout video : {videos[0]}")
    print(f"Copied final video to   : {final_video}")
    print("=" * 100)


def main():
    parser = argparse.ArgumentParser(
        description="Terminal-driven natural-language VLA manipulation demo for LeRobot + LIBERO."
    )

    parser.add_argument("--suite", default="all", help="Suite to search: libero_object, libero_goal, libero_spatial, libero_10, or all.")
    parser.add_argument("--instruction", default=None, help="Natural-language command to execute.")
    parser.add_argument("--list", action="store_true", help="List supported commands for the selected suite.")
    parser.add_argument("--list-all", action="store_true", help="List all supported commands from all selected suites.")
    parser.add_argument("--device", default="cuda", help="cuda or cpu.")
    parser.add_argument("--episodes", type=int, default=1)
    parser.add_argument("--policy", default="lerobot/pi05_libero_finetuned")
    parser.add_argument("--policy-tag", default="pi05")
    parser.add_argument("--dtype", default=None, help="Optional dtype, for example float16.")
    parser.add_argument("--project-dir", default="/kaggle/working/vla_robotic_manipulation")

    args = parser.parse_args()

    if args.suite == "all" or args.list_all:
        rows = load_all_tasks()
    else:
        rows = load_tasks_for_suite(args.suite)

    results_dir = Path(args.project_dir) / "results"
    save_task_list(rows, results_dir)

    if args.list or args.list_all:
        print_tasks(rows)
        return

    if not args.instruction:
        print("ERROR: Provide --instruction or use --list / --list-all.")
        raise SystemExit(1)

    matched_row, score = match_instruction(rows, args.instruction)
    print(f"Matched instruction with confidence score: {score:.3f}")

    run_eval(matched_row, args)


if __name__ == "__main__":
    main()

In [ ]:
!ls -lh /kaggle/working/vla_robotic_manipulation/scripts

In [ ]:
%cd /kaggle/working/vla_robotic_manipulation

!printf "N\n" | \
/kaggle/working/miniforge3/envs/lerobot/bin/python \
scripts/run_vla.py \
--list-all

In [ ]:
%cd /kaggle/working/vla_robotic_manipulation

!printf "N\n" | \
/kaggle/working/miniforge3/envs/lerobot/bin/python \
scripts/run_vla.py \
--suite libero_10 \
--instruction "put the yellow and white mug in the microwave and close it" \
--device cuda

/kaggle/working/vla_robotic_manipulation
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Saved task CSV: /kaggle/working/vla_robotic_manipulation/results/supported_libero_tasks.csv
Saved task TXT: /kaggle/working/vla_robotic_manipulation/results/supported_libero_tasks.txt
Matched instruction with confidence score: 1.000
Terminal-driven VLA robotic manipulation demo
Typed instruction : put the yellow and white mug in the microwave and close it
Matched suite     : libero_10
Matched task ID   : 9
Supported command : put the yellow and white mug in the microwave and close it
Task name         : KITCHEN_SCENE6_put_the_yellow_and_white_mug_in_the_microwave_and_close_it
Policy            : lerobot/pi05_libero_finetuned
Device            : cuda
Output directory  : /kaggle/working/vla_robotic_manipulation/outputs/eval/pi05_libero_10_task_9_cuda
Log file          : /kaggle/working/vla_robotic_manipulation/logs/pi05_libero_10_task_9_cuda.log
Final video       : /kaggle/working/vla_robotic

In [ ]:
!find /kaggle/working/vla_robotic_manipulation/media/final_demo -name "*.mp4" -type f
!ls -lh /kaggle/working/vla_robotic_manipulation/media/final_demo

In [ ]:
from IPython.display import Video, display
import glob

videos = sorted(glob.glob("/kaggle/working/vla_robotic_manipulation/media/final_demo/pi05_libero_10_task_9_cuda.mp4"))
print(videos[-1])
display(Video(videos[-1], embed=True, width=640))